In [1]:
# Cell 1: Imports
import os
import ast
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from shapely import wkt
from tqdm import tqdm
import matplotlib.pyplot as plt

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Cell 2: Config
class Config:
    CSV_PATH = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping/AOI_11_Rotterdam/SummaryData/SN6_Train_AOI_11_Rotterdam_Buildings.csv' # Replace with your CSV path
    OPTICAL_DIR = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping/AOI_11_Rotterdam/PS-RGB' # Replace with Optical dir
    SAR_DIR = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping/AOI_11_Rotterdam/SAR-Intensity'         # Replace with SAR dir
    
    IMG_SIZE = 512 # As per SpaceNet 6 standards mentioned in the paper
    BATCH_SIZE = 4
    EPOCHS = 10
    LR = 1e-4
    NUM_SAMPLES = 100 # Limit for quick training/testing
    
    # Loss weights as defined in the paper
    LAMBDA_FOOTPRINT = 1.0
    LAMBDA_HEIGHT = 0.5

In [13]:
# Cell 3: Dataset Class (With strict NaN handling)
class SpaceNetDataset(Dataset):
    def __init__(self, csv_path, optical_dir, sar_dir, img_size=512, limit=None):
        self.optical_dir = optical_dir
        self.sar_dir = sar_dir
        self.img_size = img_size
        
        df = pd.read_csv(csv_path)
        self.grouped = df.groupby('ImageId')
        self.image_ids = list(self.grouped.groups.keys())
        
        if limit:
            self.image_ids = self.image_ids[:limit]
            
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size, img_size), antialias=True)
        ])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        group = self.grouped.get_group(img_id)
        
        opt_path = os.path.join(self.optical_dir, f"SN6_Train_AOI_11_Rotterdam_PS-RGB_{img_id}.tif") 
        sar_path = os.path.join(self.sar_dir, f"SN6_Train_AOI_11_Rotterdam_SAR-Intensity_{img_id}.tif")

        
        # --- NEW: Helpful Debugger ---
        if not os.path.exists(opt_path):
            print(f"\nMissing File! Looked for: {opt_path}")
        
        if not os.path.exists(opt_path):
            opt_img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
            sar_img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            opt_img = cv2.imread(opt_path)[..., ::-1] 
            sar_img = cv2.imread(sar_path, cv2.IMREAD_GRAYSCALE)
            if len(sar_img.shape) == 2:
                sar_img = np.expand_dims(sar_img, axis=-1)
                
        mask_footprint = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        mask_height = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        
        for _, row in group.iterrows():
            poly_wkt = row['PolygonWKT_Pix']
            height = row['Mean_Building_Height']
            
            if pd.isna(poly_wkt) or poly_wkt.strip() == 'POLYGON EMPTY':
                continue
                
            # --- THE FIX: Catch missing heights in the CSV ---
            if pd.isna(height):
                height = 0.0 
                
            try:
                poly = wkt.loads(poly_wkt)
                coords = np.array(poly.exterior.coords, dtype=np.int32)
                cv2.fillPoly(mask_footprint, [coords], 1.0)
                # Ensure height is strictly a float and not NaN
                cv2.fillPoly(mask_height, [coords], float(height))
            except Exception as e:
                pass 
                
        opt_tensor = self.transform(opt_img)
        if sar_img.shape[-1] == 1:
            sar_img = np.repeat(sar_img, 3, axis=-1)
        sar_tensor = self.transform(sar_img)
        
        mask_footprint = torch.tensor(mask_footprint, dtype=torch.long)
        mask_height = torch.tensor(mask_height, dtype=torch.float32).unsqueeze(0)
        
        # --- THE FIX: Final safeguard before returning ---
        mask_height = torch.nan_to_num(mask_height, nan=0.0)
        
        return opt_tensor, sar_tensor, mask_footprint, mask_height, img_id

In [7]:
# Cell 4: Modules (Encoders & Fusion)
class PretrainedCNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        self.initial = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool) # F0
        self.layer1 = resnet.layer1 # F1
        self.layer2 = resnet.layer2 # F2
        self.layer3 = resnet.layer3 # F3
        self.layer4 = resnet.layer4 # Fout (1/32 resolution, we will upsample to 1/16 to match paper)
        
        # The paper uses 1/16 as the lowest resolution. We'll adjust layer4 output.
        self.up_align = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)

    def forward(self, x):
        f0 = self.initial(x)  # H/4 (ResNet skips H/2 due to initial conv stride + maxpool)
        f1 = self.layer1(f0)  # H/4
        f2 = self.layer2(f1)  # H/8
        f3 = self.layer3(f2)  # H/16
        f4 = self.layer4(f3)  # H/32
        f_out = self.up_align(f4) # H/16, Channels: 256
        return f0, f1, f2, f_out

class CrossFusionBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()
        self.norm1_sar = nn.LayerNorm(embed_dim)
        self.norm1_opt = nn.LayerNorm(embed_dim)
        
        self.cross_attn_sar = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.cross_attn_opt = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        
        self.norm2_sar = nn.LayerNorm(embed_dim)
        self.norm2_opt = nn.LayerNorm(embed_dim)
        
        self.mlp_sar = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(), nn.Linear(embed_dim * 2, embed_dim)
        )
        self.mlp_opt = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(), nn.Linear(embed_dim * 2, embed_dim)
        )

    def forward(self, sar_feat, opt_feat):
        # Reshape (B, C, H, W) -> (B, H*W, C)
        B, C, H, W = sar_feat.shape
        sar_flat = sar_feat.flatten(2).transpose(1, 2)
        opt_flat = opt_feat.flatten(2).transpose(1, 2)
        
        # Norm
        sar_norm = self.norm1_sar(sar_flat)
        opt_norm = self.norm1_opt(opt_flat)
        
        # Cross Attention: Modality acts as Query, the OTHER modality acts as Key/Value
        attn_sar, _ = self.cross_attn_sar(sar_norm, opt_norm, opt_norm)
        sar_out = sar_flat + attn_sar
        
        attn_opt, _ = self.cross_attn_opt(opt_norm, sar_norm, sar_norm)
        opt_out = opt_flat + attn_opt
        
        # MLP
        sar_out = sar_out + self.mlp_sar(self.norm2_sar(sar_out))
        opt_out = opt_out + self.mlp_opt(self.norm2_opt(opt_out))
        
        # Reshape back to (B, C, H, W)
        sar_out = sar_out.transpose(1, 2).view(B, C, H, W)
        opt_out = opt_out.transpose(1, 2).view(B, C, H, W)
        
        return sar_out, opt_out

In [8]:
# Cell 5: Decoder and Full Architecture
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Ensure spatial dimensions match if rounding issues occur
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class FusionHeightNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_opt = PretrainedCNNEncoder()
        self.enc_sar = PretrainedCNNEncoder()
        
        self.cross_fusion = CrossFusionBlock(embed_dim=256)
        
        # Joint fusion of optical and SAR for the shared lowest resolution
        self.fusion_conv = nn.Conv2d(512, 256, 1) 
        
        # Footprint Decoder (using optical skips for sharp edges)
        self.foot_dec1 = DecoderBlock(256, 128, 128) # Uses f2
        self.foot_dec2 = DecoderBlock(128, 64, 64)   # Uses f1
        self.foot_dec3 = DecoderBlock(64, 64, 64)    # Uses f0
        self.foot_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Upsample(scale_factor=4, mode='bilinear'), # Upscale remaining H/4 to H
            nn.Conv2d(32, 2, 1) # 2 Classes: Background, Footprint
        )
        
        # Height Decoder (using SAR skips for structural geometry)
        self.height_dec1 = DecoderBlock(256, 128, 128)
        self.height_dec2 = DecoderBlock(128, 64, 64)
        self.height_dec3 = DecoderBlock(64, 64, 64)
        self.height_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Upsample(scale_factor=4, mode='bilinear'),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid() # Restricts output to 0-1 (Assumes GT height is normalized!)
        )

    def forward(self, opt, sar):
        o_f0, o_f1, o_f2, o_fout = self.enc_opt(opt)
        s_f0, s_f1, s_f2, s_fout = self.enc_sar(sar)
        
        # Multi-Level Cross Fusion at the bottleneck
        s_fused, o_fused = self.cross_fusion(s_fout, o_fout)
        
        # Combine for decoder input
        f_fusion = self.fusion_conv(torch.cat([o_fused, s_fused], dim=1))
        
        # Footprint Branch
        d_foot = self.foot_dec1(f_fusion, o_f2)
        d_foot = self.foot_dec2(d_foot, o_f1)
        d_foot = self.foot_dec3(d_foot, o_f0)
        pred_footprint = self.foot_head(d_foot)
        
        # Height Branch
        d_height = self.height_dec1(f_fusion, s_f2)
        d_height = self.height_dec2(d_height, s_f1)
        d_height = self.height_dec3(d_height, s_f0)
        base_height = self.height_head(d_height)
        
        # Semantic Refinement (Paper Eq 16)
        # Instead of non-differentiable argmax, we use the probability of the footprint class
        footprint_prob = F.softmax(pred_footprint, dim=1)[:, 1:2, :, :] 
        pred_height = base_height * footprint_prob
        
        return pred_footprint, pred_height

In [9]:
# Cell 6: Losses
def dice_loss(pred, target, smooth=1e-6):
    pred = F.softmax(pred, dim=1)[:, 1, :, :] # Take footprint class prob
    intersection = (pred * target).sum(dim=(1, 2))
    union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()

def fusion_loss(pred_foot, pred_height, gt_foot, gt_height, config):
    # Footprint Loss (CrossEntropy + Dice)
    ce_loss = F.cross_entropy(pred_foot, gt_foot)
    d_loss = dice_loss(pred_foot, gt_foot)
    L_footprint = ce_loss + d_loss
    
    # Height Loss (Smooth L1)
    # Note: Ensure your gt_height is normalized between 0-1 for this to work correctly with Sigmoid
    # E.g., gt_height = gt_height / MAX_HEIGHT
    L_height = F.smooth_l1_loss(pred_height, gt_height)
    
    # Total
    L_total = (config.LAMBDA_FOOTPRINT * L_footprint) + (config.LAMBDA_HEIGHT * L_height)
    return L_total, L_footprint, L_height

In [14]:
# MODIFIED Cell 7: Execution with Stability Fixes
from torch.utils.data import random_split

if __name__ == '__main__':
    # Initialize Dataset
    full_dataset = SpaceNetDataset(
        csv_path=Config.CSV_PATH, 
        optical_dir=Config.OPTICAL_DIR, 
        sar_dir=Config.SAR_DIR, 
        img_size=Config.IMG_SIZE,
        limit=Config.NUM_SAMPLES
    )
    
    # --- NEW: Train/Test Split (80/20) ---
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    # Sanity Check: Ensure data is actually loading
    sample_opt, sample_sar, _, _, _ = train_dataset[0]
    if sample_opt.max() == 0:
        print("⚠️ WARNING: Your Optical images are completely black. Check OPTICAL_DIR path!")
    if sample_sar.max() == 0:
        print("⚠️ WARNING: Your SAR images are completely black. Check SAR_DIR path!")
    
    # Initialize Model
    model = FusionHeightNet().to(device)
    
    # --- FIX 1: Use AdamW instead of SGD for Transformer stability ---
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=1e-4)
    
    MAX_HEIGHT = 50.0 
    
    print(f"Starting Training on {train_size} images, testing on {test_size} images...")
    for epoch in range(Config.EPOCHS):
        model.train()
        epoch_loss = 0.0
        
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        for opt, sar, gt_foot, gt_height, _ in progress:
            opt = opt.to(device)
            sar = sar.to(device)
            gt_foot = gt_foot.to(device)
            gt_height = (gt_height / MAX_HEIGHT).to(device)
            
            # --- NEW SAFETY CHECK ---
            if torch.isnan(gt_height).any() or torch.isnan(opt).any() or torch.isnan(sar).any():
                continue # Skip this batch, the data is corrupted
            # ------------------------
            
            optimizer.zero_grad()
            pred_foot, pred_height = model(opt, sar)
            # ... rest of your code ...
            
            loss, l_foot, l_height = fusion_loss(pred_foot, pred_height, gt_foot, gt_height, Config)
            loss.backward()
            
            # --- FIX 2: Gradient Clipping to prevent NaN explosions ---
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_loss += loss.item()
            progress.set_postfix({'Total Loss': loss.item()})
            
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")
        
        save_path = f'fusion_height_net_epoch_{epoch+1}.pth'
        torch.save(model.state_dict(), save_path)

[ WARN:0@419.374] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@419.374] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@419.374] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@419.374] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@419.374] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 42113 (0xa481) encountered
[ WARN:0@419.432] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@419.432] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@419.432] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) enco

AttributeError: 'NoneType' object has no attribute 'shape'

In [14]:
# MODIFIED Cell 8: Percentage-Based Metrics Logic
import scipy.ndimage as ndimage
import numpy as np

def calculate_instance_metrics(pred_foot_prob, pred_height_map, gt_foot, gt_height_map, height_tolerance_pct=0.25):
    """
    Calculates building counts and height accuracy within a PERCENTAGE tolerance.
    """
    # 1. Binarize the predicted footprint (Threshold at 0.5)
    pred_mask = (pred_foot_prob > 0.5).astype(np.uint8)
    gt_mask = gt_foot.astype(np.uint8)
    
    # 2. Extract distinct building instances using connected components
    pred_labeled, pred_num_features = ndimage.label(pred_mask)
    gt_labeled, gt_num_features = ndimage.label(gt_mask)
    
    matched_buildings = 0
    height_within_tolerance_count = 0
    
    # 3. Match predicted buildings to GT buildings
    for i in range(1, pred_num_features + 1):
        # Get pixels for this specific predicted building
        pred_building_pixels = (pred_labeled == i)
        
        # Find which GT building overlaps the most with this prediction
        overlapping_gt_labels = gt_labeled[pred_building_pixels]
        overlapping_gt_labels = overlapping_gt_labels[overlapping_gt_labels > 0]
        
        if len(overlapping_gt_labels) > 0:
            # We found a match
            matched_buildings += 1
            most_common_gt_label = np.bincount(overlapping_gt_labels).argmax()
            gt_building_pixels = (gt_labeled == most_common_gt_label)
            
            # Calculate average height for this building instance
            avg_pred_height = np.mean(pred_height_map[pred_building_pixels])
            avg_gt_height = np.mean(gt_height_map[gt_building_pixels])
            
            # --- PERCENTAGE LOGIC ---
            if avg_gt_height > 0:
                allowed_error = avg_gt_height * height_tolerance_pct
            else:
                allowed_error = 1.0 # Fallback for 0-height flat ground errors
                
            if abs(avg_pred_height - avg_gt_height) <= allowed_error:
                height_within_tolerance_count += 1
                
    return gt_num_features, pred_num_features, matched_buildings, height_within_tolerance_count

In [15]:
# MODIFIED Cell 9: Per-Image Evaluation & Summaries
def test_model(model_path, test_loader, height_tolerance_pct=0.25, max_h=50.0):
    print(f"Loading model from {model_path} for testing...\n")
    
    model = FusionHeightNet().to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    total_actual_buildings = 0
    total_pred_buildings = 0
    total_matched_buildings = 0
    total_accurate_height_buildings = 0
    
    print(f"{'Image ID':<45} | {'Actual':<6} | {'Detected':<8} | {'Matched':<7} | {'Correct Hgt':<11}")
    print("-" * 88)
    
    with torch.no_grad():
        # Notice we unpack `img_id` here now
        for opt, sar, gt_foot, gt_height, img_id in test_loader:
            opt, sar = opt.to(device), sar.to(device)
            
            # Get Predictions
            pred_foot, pred_height = model(opt, sar)
            
            # Convert to numpy
            pred_foot_prob = F.softmax(pred_foot, dim=1)[0, 1, :, :].cpu().numpy() 
            pred_height_map = (pred_height[0, 0, :, :].cpu().numpy()) * max_h 
            gt_foot_np = gt_foot[0].cpu().numpy()
            gt_height_map = gt_height[0, 0, :, :].cpu().numpy() 
            
            # Calculate metrics
            actual, pred_count, matched, height_accurate = calculate_instance_metrics(
                pred_foot_prob, pred_height_map, gt_foot_np, gt_height_map, height_tolerance_pct=height_tolerance_pct
            )
            
            # Print PER IMAGE stats
            img_name = img_id[0]
            print(f"{img_name:<45} | {actual:<6} | {pred_count:<8} | {matched:<7} | {height_accurate:<11}")
            
            # Accumulate totals
            total_actual_buildings += actual
            total_pred_buildings += pred_count
            total_matched_buildings += matched
            total_accurate_height_buildings += height_accurate

    # Print Final Summary
    print("\n" + "="*50)
    print("      FINAL TEST METRICS SUMMARY      ")
    print("="*50)
    print(f"Total Actual Buildings in GT: {total_actual_buildings}")
    print(f"Total Buildings Detected:     {total_pred_buildings}")
    print(f"Successfully Matched:         {total_matched_buildings}")
    print(f"Heights within +/- {height_tolerance_pct*100}%:      {total_accurate_height_buildings}")
    print("-" * 50)
    
    if total_actual_buildings > 0:
        detection_rate = (total_matched_buildings / total_actual_buildings) * 100
        print(f"Detection Rate:               {detection_rate:.2f}%")
        
    if total_matched_buildings > 0:
        height_acc_rate = (total_accurate_height_buildings / total_matched_buildings) * 100
        print(f"Height Accuracy (within %):   {height_acc_rate:.2f}%")
    print("="*50)

# Execute the test
if __name__ == '__main__':
    # 0.25 = 25% tolerance
    TOLERANCE_PCT = 0.25 
    final_model_path = f'fusion_height_net_epoch_{Config.EPOCHS}.pth'
    
    test_model(final_model_path, test_loader, height_tolerance_pct=TOLERANCE_PCT, max_h=MAX_HEIGHT)

Loading model from fusion_height_net_epoch_10.pth for testing...


Evaluating: 100%|██████████| 20/20 [00:01<00:00, 19.71it/s]


--- FINAL TEST METRICS ---
Total Actual Buildings in GT: 392
Total Buildings Detected: 40
Successfully Matched Buildings: 3
Matched Buildings with Height within +/- 3.0m: 1
Detection Rate: 0.77%
Height Accuracy (within offset): 33.33%
